In [1]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import time
import gget
import scipy
from scipy.sparse import csr_matrix
import anndata as an
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import random
from importlib import reload
import warnings
import ot
from scipy.spatial.distance import pdist, squareform
from matplotlib.colors import ListedColormap


In [2]:
%%time 
# 1MB resolution
resolution = 1000000

fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/anndata/population_mESC_{resolution}_features.h5ad"

adata = sc.read_h5ad(fpath)

sc.logging.print_memory_usage()

adata

Memory usage: current 2.82 GB, difference +2.82 GB
CPU times: user 12.3 s, sys: 3.07 s, total: 15.4 s
Wall time: 47.3 s


/scratch/indikar_root/indikar1/sgolts/conda-envs/pore_c/lib/python3.13/site-packages/anndata/logging.py:57: FutureWarning: The specified parameters ('newline',) are no longer positional. Please specify them like `newline=False`
  print(format_memory_usage(get_memory_usage(), msg, newline))


AnnData object with n_obs × n_vars = 2579 × 2756467
    obs: 'bin_index', 'bin_start', 'bin_end', 'bin', 'chrom', 'chrom_bin', 'degree', 'genes', 'n_genes', 'ATACSeq_1', 'ATACSeq_2', 'ATACSeq_3', 'CTCF', 'H3K27ac', 'H3K27me3', 'RNA_1', 'RNA_2', 'RNA_3', 'RNA_4', 'RNA_5', 'RNA_6', 'PolII'
    var: 'read_index', 'basename', 'mean_mapq', 'median_mapq', 'n_chromosomes', 'order', 'n_bins', 'read_length_bp', 'genes', 'n_genes'
    uns: 'base_resolution', 'chrom_sizes', 'gdf', 'gene_map', 'intervals'
    layers: 'H'

In [15]:
adata.obs["n_genes"].head()

bin_name
chr9:121    41
chr19:26    11
chr4:127    16
chr8:21     47
chr10:57    14
Name: n_genes, dtype: int64

In [4]:
H = adata.X  # keep sparse

In [5]:
print(f"raw shape = {H.shape}")

raw shape = (2579, 2756467)


In [8]:
# ---- remove outlier bins ----
s = np.asarray(H.sum(axis=1)).ravel()

q1, q3 = np.quantile(s, [0.25, 0.75])
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

keep = (s >= low) & (s <= high)
H = H[keep]

# ---- drop newly created singletons ----
col_sums = np.asarray(H.sum(axis=0)).ravel()
H = H[:, col_sums >= 2]

# ---- drop disgusting hyperedges ----
col_sums = np.asarray(H.sum(axis=0)).ravel()
H = H[:, col_sums <= 10]

In [9]:
print(f"filtered shape = {H.shape}")

filtered shape = (2335, 2039865)


In [13]:
print(H[5,4])

0


In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 3, 3.25

x = H.sum(0)
sns.histplot(x, discrete=True, bins=range(int(x.min()), int(x.max())+2))
plt.xticks(range(int(x.min()), int(x.max())+1))

plt.ylabel('Hyperedges')
plt.xlabel('Order')

sns.despine()

In [2]:
fpath = "utilities/calculations/outputs/edges_curvature.csv"

curv = pd.read_csv(fpath)

curv.head()

/tmp/ipykernel_122183/333049772.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  curv = pd.read_csv(fpath)


,node_u_id,node_u_label,node_u_type,node_v_id,node_v_label,node_v_type,read_label,triangles,edge_curvature
0,0,121,locus,1933321,2606795,read,2606795,0,-3055
1,0,121,locus,1654792,2161981,read,2161981,5,-3042
2,0,121,locus,1559745,2032381,read,2032381,1,-3059
3,0,121,locus,648840,808648,read,808648,2,-3055
4,0,121,locus,422154,519394,read,519394,19,-3129


In [7]:
curv["read_label"].nunique()

28748488